# Python Day 3 강의 노트북 — 클래스·파일·시스템 연동

**HYUNDAI AI Insight Campus 온디바이스 AI · 프로그래밍 언어(Python) 3/4 · 9/4(금)**

| 교시 | 주제 | 애니메이션 |
|---|---|---|
| 1 | 클래스 기초 — dict 를 졸업하기 | D3-1 클래스·인스턴스 모델 |
| 2 | 클래스 심화 — repr·클래스 변수·상속 | D3-1 ⑤⑥ · D3-2 상속·메서드 탐색 |
| 3 | 파일 입출력 | D3-3 with open 파일 흐름 |
| 4 | JSON 과 CSV | D3-4 JSON 직렬화 왕복 |
| 5 | 시스템 연동 | D3-5 시리얼 스트림 · 04 파이프라인 |
| 6 | 종합 설계 — SensorLogger | D3-1 · D3-4 · 03(메서드 호출 스택) |
| 7 | 종합 실습 — 저장되는 센서 로거 | D3-4 · D2-5 트리 뷰어 |
| 8 | 정리·퀴즈 | — |

### 이 노트북 사용법 (Day 1·2와 동일)
- **▶ 예제** 셀은 그대로 실행하며 출력을 함께 읽습니다.
- **✏️ 빈칸 채우기** 셀은 `____` 를 채워 실행합니다. 채우기 전에 실행하면 오류가 나는 것이 정상입니다.
- **✏️ 괄호 넣기**는 ( ) 에 들어갈 말을 생각한 뒤 정답 셀로 확인합니다.
- **🔒 정답 보기** 셀은 단독으로 실행됩니다. 준비 셀 없이도 동작합니다. 강사가 신호를 준 뒤 실행하세요.

In [ ]:
#@title ⚙️ 준비 — 가장 먼저 실행 (공용 모듈·Day 2 도구·데이터)
import math, random, os, sys, json, csv, subprocess, time
from collections import Counter
from pathlib import Path

# Day 2 에서 만든 도구 (제공)
class FrameError(Exception):
    pass

def parse_line(line):
    """$T=..,H=..,D=..* → (float, int, int). 검증 실패 시 FrameError."""
    line = line.strip()
    if not (line.startswith("$") and line.endswith("*")):
        raise FrameError(f"손상 프레임: {line!r}")
    d = {}
    for f in line[1:-1].split(","):
        k, v = f.split("=")
        d[k] = v
    return (float(d["T"]), int(d["H"]), int(d["D"]))

DAY2_READINGS = [
    {"id": "ESP32-01", "T": 26.4, "H": 55, "D": 132},
    {"id": "ESP32-01", "T": 31.2, "H": 61, "D": 118},
    {"id": "ESP32-02", "T": 36.9, "H": 70, "D": 150},
    {"id": "ESP32-02", "T": 29.8, "H": 66, "D": 140},
    {"id": "ESP32-03", "T": 25.1, "H": 48, "D": 210},
    {"id": "ESP32-03", "T": 33.4, "H": 52, "D": 190},
]
STREAM = ["$T=25.3,H=60,D=120*", "", "T=99.9,H=0",
          "$T=36.9,H=70,D=150*", "$T=2", "$T=26.1,H=58,D=118*"]
print("준비 완료.")

---
# 1교시 · 클래스 기초 — dict 를 졸업하기 (09:00–09:50)

**학습 목표**
- dict 레코드의 한계를 체감하고 class · `__init__` · 메서드를 정의한다
- self = "지금 이 인스턴스" 를 설명한다

🎬 **D3-1 클래스·인스턴스 메모리 모델** ① 설계도 → ② 인스턴스 → ③ 독립 → ④ self → ⑦ dict 비교

### ▶ 예제 1-1 · 어제 코드의 불편함
dict 레코드는 키 오타를 실행 전에 못 잡고, 판정 함수는 데이터와 떨어져 삽니다.

In [ ]:
r = {"id": "ESP32-01", "T": 25.3}
def judge(rec):
    return "ALERT" if rec["T"] > 35 else "OK"

print(judge(r))
try:
    r["Temp"]        # 오타 — 실행해야 터지고, 어떤 키가 맞는지 힌트가 없다
except KeyError as e:
    print("KeyError:", e)

### ▶ 예제 1-2 · 첫 클래스 — 데이터와 동작을 한 상자에

In [ ]:
class SensorReading:
    def __init__(self, node_id, t, h, d):
        self.node_id = node_id
        self.t = t
        self.h = h
        self.d = d

    def is_alert(self):
        return self.t > 35

r1 = SensorReading("ESP32-01", 25.3, 60, 120)
r2 = SensorReading("ESP32-02", 36.9, 70, 150)
print(r1.t, r1.is_alert())
print(r2.node_id, r2.is_alert())

### ▶ 예제 1-3 · self 의 정체
`r1.is_alert()` 는 `SensorReading.is_alert(r1)` 의 줄임입니다. 🎬 D3-1 ④

In [ ]:
print(r1.is_alert(), SensorReading.is_alert(r1))   # 완전히 같은 호출
r2.t = 37.5
print(r1.t, r2.t)     # 인스턴스 속성은 서로 독립 (Day 2 참조 공유와 대비)

### ▶ 예제 1-4 · 오타의 운명 비교

In [ ]:
try:
    r1.tmp
except AttributeError as e:
    print("AttributeError:", e)   # 어떤 클래스에 어떤 속성이 없는지 즉시 알려준다

#### ✏️ 빈칸 채우기 1-1
`____` 를 채운 뒤 실행하세요. 정답은 아래 **정답 보기** 셀을 실행하면 나타납니다.

In [ ]:
# GPIO 핀 클래스: 이름과 번호를 저장하고 info() 로 요약
class Pin:
    def __init__(____, name, number):
        ____.name = name
        self.number = number
    def info(self):
        return f"{self.name} → GPIO {self.number}"

led = Pin("LED", 3)
print(led.info())            # LED → GPIO 3

In [ ]:
#@title 🔒 정답 보기 1-1 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAxLTEg4pSA4pSACmRlZiBfX2luaXRfXyhzZWxmLCBuYW1lLCBudW1iZXIpOgogICAgc2Vs"
    "Zi5uYW1lID0gbmFtZQ=="
).decode("utf-8"))

#### ✏️ 빈칸 채우기 1-2
`____` 를 채운 뒤 실행하세요. 정답은 아래 **정답 보기** 셀을 실행하면 나타납니다.

In [ ]:
# Day 1 판정 규칙을 메서드로
class SensorReading:
    def __init__(self, node_id, t, h=0, d=0):
        self.node_id = node_id; self.t = t; self.h = h; self.d = d
    def level(____):
        if self.t < 0: return "FREEZE"
        elif self.t < 20: return "COLD"
        elif ____.t < 30: return "NORMAL"
        elif self.t < 40: return "WARM"
        return "ALERT"

print([SensorReading("X", t).level() for t in (-5, 25, 45)])   # ['FREEZE', 'NORMAL', 'ALERT']

In [ ]:
#@title 🔒 정답 보기 1-2 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAxLTIg4pSA4pSACmRlZiBsZXZlbChzZWxmKToKLi4uCmVsaWYgc2VsZi50IDwgMzA6IHJl"
    "dHVybiAiTk9STUFMIg=="
).decode("utf-8"))

#### ✏️ 괄호 넣기 1-3
1. 클래스는 (　　　)이고, 인스턴스는 그 설계도로 찍어낸 실제 (　　　)이다.
2. `r1.is_alert()` 는 `SensorReading.is_alert(　　　)` 의 줄임이다.
3. self 는 특별한 키워드가 아니라 (　　　)의 관례적 이름이다.
4. 없는 속성 `r1.tmp` 를 읽으면 (　　　) 예외가 난다.

In [ ]:
#@title 🔒 정답 보기 1-3 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAxLTMg4pSA4pSACjEuIOyEpOqzhOuPhCAvIOuNsOydtO2EsAoyLiByMQozLiDssqsg66ek"
    "6rCc67OA7IiYCjQuIEF0dHJpYnV0ZUVycm9y"
).decode("utf-8"))

---
# 2교시 · 클래스 심화 — repr·클래스 변수·상속 (10:00–10:50)

🎬 **D3-1** ⑤ 조회 경로 · ⑥ 전파 → **D3-2 상속과 메서드 탐색** ①~⑤

### ▶ 예제 2-1 · __repr__ — 디버깅의 절반

In [ ]:
class SensorReading:
    ALERT_T = 35                       # 클래스 변수 (설계도에 한 부)

    def __init__(self, node_id, t, h=0, d=0):
        self.node_id = node_id; self.t = t; self.h = h; self.d = d

    def __repr__(self):
        return f"Reading({self.node_id}, T={self.t})"

    def is_alert(self):
        return self.t > self.ALERT_T

r = SensorReading("ESP32-01", 25.3)
print(r)              # __repr__ 가 없으면 <object at 0x...> 로 보인다
print([r, SensorReading("ESP32-02", 36.9)])

### ▶ 예제 2-2 · 클래스 변수 — 조회 순서와 전파
🎬 D3-1 ⑤ (인스턴스 → 클래스) · ⑥ (전체 전파)

In [ ]:
a = SensorReading("A", 32.0)
b = SensorReading("B", 33.0)
print(a.is_alert(), b.is_alert())      # False False (기준 35)

SensorReading.ALERT_T = 30             # 설계도 한 곳을 바꾸면
print(a.is_alert(), b.is_alert())      # True True — 모두에게 전파
SensorReading.ALERT_T = 35             # 복구

### ▶ 예제 2-3 · 상속 — 공통은 부모에, 다른 것만 자식에
🎬 D3-2 ③ 오버라이드 · ④ super

In [ ]:
class CalibratedReading(SensorReading):
    def __init__(self, node_id, t, offset):
        super().__init__(node_id, t)   # 부모의 초기화 먼저
        self.offset = offset

    def is_alert(self):                # 오버라이드
        return (self.t + self.offset) > self.ALERT_T

c = CalibratedReading("ESP32-01", 34.0, 2.0)
print(c)                # __repr__ 는 부모에서 찾아 실행 (자식→부모 탐색)
print(c.is_alert())     # True — 자식 버전에서 멈춤 (34+2 > 35)

### ▶ 예제 2-4 · ⚠ super 누락 사고
🎬 D3-2 ⑤

In [ ]:
class BadReading(SensorReading):
    def __init__(self, node_id, t, offset):
        self.offset = offset           # super() 를 잊었다!

bad = BadReading("X", 25.3, 0.5)
try:
    bad.t
except AttributeError as e:
    print("AttributeError:", e)        # 부모가 만들던 t 가 없다

#### ✏️ 빈칸 채우기 2-1
`____` 를 채운 뒤 실행하세요. 정답은 아래 **정답 보기** 셀을 실행하면 나타납니다.

In [ ]:
# print(r) 이 "Reading(ESP32-01, T=25.3)" 으로 보이게
class SensorReading:
    def __init__(self, node_id, t):
        self.node_id = node_id; self.t = t
    def ____(self):
        return f"Reading({self.node_id}, T={____})"

print(SensorReading("ESP32-01", 25.3))

In [ ]:
#@title 🔒 정답 보기 2-1 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAyLTEg4pSA4pSACmRlZiBfX3JlcHJfXyhzZWxmKToKICAgIHJldHVybiBmIlJlYWRpbmco"
    "e3NlbGYubm9kZV9pZH0sIFQ9e3NlbGYudH0pIg=="
).decode("utf-8"))

#### ✏️ 빈칸 채우기 2-2
`____` 를 채운 뒤 실행하세요. 정답은 아래 **정답 보기** 셀을 실행하면 나타납니다.

In [ ]:
# 보정값을 가진 자식 클래스
class CalibratedReading(SensorReading):
    def __init__(self, node_id, t, offset):
        ____().__init__(node_id, t)
        self.offset = offset
    def calibrated(self):
        return self.t + ____.offset

c = CalibratedReading("ESP32-01", 34.0, 2.0)
print(c.calibrated())        # 36.0

In [ ]:
#@title 🔒 정답 보기 2-2 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAyLTIg4pSA4pSACnN1cGVyKCkuX19pbml0X18obm9kZV9pZCwgdCkKcmV0dXJuIHNlbGYu"
    "dCArIHNlbGYub2Zmc2V0"
).decode("utf-8"))

#### ✏️ 괄호 넣기 2-3
1. `r1.ALERT_T` 조회는 (　　　) 에서 먼저 찾고, 없으면 (　　　) 로 올라간다.
2. 자식이 같은 이름의 메서드를 정의하면 탐색이 자식에서 멈춘다 — 이것을 (　　　)라 한다.
3. 자식 `__init__` 첫 줄의 `super().__init__(...)` 은 (　　　)의 초기화를 먼저 실행한다.
4. 클래스 변수로 리스트를 두면 모든 인스턴스가 (　　　)하므로 위험하다.

In [ ]:
#@title 🔒 정답 보기 2-3 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAyLTMg4pSA4pSACjEuIOyduOyKpO2EtOyKpCAvIO2BtOuemOyKpAoyLiDsmKTrsoTrnbzs"
    "nbTrk5wKMy4g67aA66qoCjQuIOqzteycoA=="
).decode("utf-8"))

---
# 3교시 · 파일 입출력 (11:00–11:50)

🎬 **D3-3 with open 파일 흐름** — 특히 ① "w 가 지우는 순간" · ④ close 없이

### ▶ 예제 3-1 · 쓰기 3박자와 모드
w 는 여는 순간 기존 내용을 지웁니다. 이어쓰기는 a.

In [ ]:
with open("sensor_log.txt", "w", encoding="utf-8") as f:   # 새로 쓰기
    f.write("T=25.3,H=60\n")
with open("sensor_log.txt", "a", encoding="utf-8") as f:   # 이어쓰기
    f.write("T=26.1,H=58\n")
print(open("sensor_log.txt", encoding="utf-8").read())

### ▶ 예제 3-2 · 한 줄씩 읽기 — strip 습관

In [ ]:
with open("sensor_log.txt", encoding="utf-8") as f:
    for line in f:
        print(repr(line))          # 줄 끝에 \n 이 붙어 있다
        print(line.strip())

### ▶ 예제 3-3 · 없는 파일 — 방어까지

In [ ]:
try:
    open("no_such_file.txt")
except FileNotFoundError as e:
    print("FileNotFoundError — 첫 실행이라면 정상적인 상황일 수 있다")

# 방어 패턴 (6교시 SensorLogger.load 의 원형)
try:
    with open("no_such_file.txt", encoding="utf-8") as f:
        lines = f.readlines()
except FileNotFoundError:
    lines = []
print(lines)

#### ✏️ 빈칸 채우기 3-1
`____` 를 채운 뒤 실행하세요. 정답은 아래 **정답 보기** 셀을 실행하면 나타납니다.

In [ ]:
# 로그 이어쓰기 함수 — write 는 개행을 자동으로 붙이지 않는다
def append_log(path, line):
    with open(path, ____, encoding="utf-8") as f:
        f.write(line + ____)

if os.path.exists("app.txt"): os.remove("app.txt")
append_log("app.txt", "$T=25.3,H=60,D=120*")
append_log("app.txt", "$T=26.1,H=58,D=118*")
print(open("app.txt", encoding="utf-8").read())

In [ ]:
#@title 🔒 정답 보기 3-1 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAzLTEg4pSA4pSACndpdGggb3BlbihwYXRoLCAiYSIsIGVuY29kaW5nPSJ1dGYtOCIpIGFz"
    "IGY6CiAgICBmLndyaXRlKGxpbmUgKyAiXG4iKQ=="
).decode("utf-8"))

#### ✏️ 빈칸 채우기 3-2
`____` 를 채운 뒤 실행하세요. 정답은 아래 **정답 보기** 셀을 실행하면 나타납니다.

In [ ]:
# 파일을 한 줄씩 읽어 Day 2 parse_line 으로 처리 (제공된 parse_line 사용)
open("mini_log.txt", "w", encoding="utf-8").write("$T=24.1,H=50,D=200*\nbroken\n$T=26.7,H=55,D=180*\n")
good = bad = 0
with open("mini_log.txt", encoding="utf-8") as f:
    for line in ____:
        try:
            parse_line(line)
        except (FrameError, ValueError):
            bad += 1
            ____
        good += 1
print(good, bad)         # 2 1

In [ ]:
#@title 🔒 정답 보기 3-2 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAzLTIg4pSA4pSACmZvciBsaW5lIGluIGY6Ci4uLgogICAgICAgIGJhZCArPSAxCiAgICAg"
    "ICAgY29udGludWU="
).decode("utf-8"))

#### ✏️ 괄호 넣기 3-3
1. `open(path, "w")` 는 여는 순간 기존 내용을 (　　　).
2. `with` 를 쓰는 가장 큰 이유는 블록이 끝나면 자동으로 (　　　) 되기 때문이다.
3. close 되지 않으면 (　　　)에 남은 내용이 디스크에 내려가지 않을 수 있다.
4. 읽은 줄 끝의 개행을 없애는 메서드는 (　　　)이다.

In [ ]:
#@title 🔒 정답 보기 3-3 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAzLTMg4pSA4pSACjEuIOyngOyatOuLpAoyLiBjbG9zZQozLiDrsoTtjbwKNC4gc3RyaXAo"
    "KQ=="
).decode("utf-8"))

---
# 4교시 · JSON 과 CSV (13:00–13:50)

🎬 **D3-4 JSON 직렬화 왕복** ①~⑥ — "s 가 붙으면 문자열, 없으면 파일"

### ▶ 예제 4-1 · dumps / loads — 문자열 왕복, 타입 복원

In [ ]:
sample = {"node": "ESP32-01", "avg_T": 26.4, "alerts": 0}
s = json.dumps(sample, ensure_ascii=False)
print(type(s), s)

back = json.loads(s)
print(back == sample, type(back["avg_T"]))   # True <class 'float'> — 타입이 돌아온다!

### ▶ 예제 4-2 · dump / load — 파일 왕복

In [ ]:
with open("stats.json", "w", encoding="utf-8") as f:
    json.dump(sample, f, indent=2, ensure_ascii=False)

with open("stats.json", encoding="utf-8") as f:
    restored = json.load(f)
print(restored == sample)
print(open("stats.json", encoding="utf-8").read())

### ▶ 예제 4-3 · ⚠ 인스턴스는 저장 불가 → to_dict 예고
🎬 D3-4 ⑤ — 6교시의 핵심 패턴

In [ ]:
class SensorReading:
    def __init__(self, node_id, t):
        self.node_id = node_id; self.t = t

r = SensorReading("ESP32-01", 25.3)
try:
    json.dumps(r)
except TypeError as e:
    print("TypeError:", e)     # JSON 이 아는 타입: dict/list/str/숫자/bool/None 뿐

### ▶ 예제 4-4 · CSV — 엑셀용 표

In [ ]:
with open("readings.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "T", "H", "D"])
    for r in DAY2_READINGS[:3]:
        w.writerow([r["id"], r["T"], r["H"], r["D"]])
print(open("readings.csv", encoding="utf-8").read())

#### ✏️ 빈칸 채우기 4-1
`____` 를 채운 뒤 실행하세요. 정답은 아래 **정답 보기** 셀을 실행하면 나타납니다.

In [ ]:
# Day 2 데이터를 파일로 저장하고 복원해 검증
with open("readings.json", "w", encoding="utf-8") as f:
    json.____(DAY2_READINGS, f, indent=2, ensure_ascii=False)
with open("readings.json", encoding="utf-8") as f:
    restored = json.____(f)
print(restored == DAY2_READINGS)     # True

In [ ]:
#@title 🔒 정답 보기 4-1 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA0LTEg4pSA4pSACmpzb24uZHVtcChEQVkyX1JFQURJTkdTLCBmLCBpbmRlbnQ9MiwgZW5z"
    "dXJlX2FzY2lpPUZhbHNlKQpyZXN0b3JlZCA9IGpzb24ubG9hZChmKQ=="
).decode("utf-8"))

#### ✏️ 빈칸 채우기 4-2
`____` 를 채운 뒤 실행하세요. 정답은 아래 **정답 보기** 셀을 실행하면 나타납니다.

In [ ]:
# 한글을 그대로 저장하려면?
s1 = json.dumps({"loc": "정문"})
s2 = json.dumps({"loc": "정문"}, ____=False)
print(s1)     # {"loc": "\uc815\ubb38"}
print(s2)     # {"loc": "정문"}

In [ ]:
#@title 🔒 정답 보기 4-2 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA0LTIg4pSA4pSACnMyID0ganNvbi5kdW1wcyh7ImxvYyI6ICLsoJXrrLgifSwgZW5zdXJl"
    "X2FzY2lpPUZhbHNlKQ=="
).decode("utf-8"))

#### ✏️ 괄호 넣기 4-3
1. `json.dump` 는 (　　　)로, `json.dumps` 는 (　　　)로 내보낸다.
2. JSON 에서 복원한 26.4 의 자료형은 (　　　)이다 — split 파싱과 달리 타입이 돌아온다.
3. 클래스 인스턴스를 그대로 dump 하면 (　　　) 예외가 난다.
4. 사람·엑셀용 표는 (　　　), 프로그램 왕복·중첩 구조는 (　　　)이 알맞다.

In [ ]:
#@title 🔒 정답 보기 4-3 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA0LTMg4pSA4pSACjEuIO2MjOydvCAvIOusuOyekOyXtAoyLiBmbG9hdAozLiBUeXBlRXJy"
    "b3IKNC4gQ1NWIC8gSlNPTg=="
).decode("utf-8"))

---
# 5교시 · 시스템 연동 (14:00–14:50)

🎬 **D3-5 시리얼 스트림 시뮬레이터** ①~⑤ · **04** (한 줄이 들어가는 곳)

### ▶ 예제 5-1 · 폴더·경로 — pathlib

In [ ]:
os.makedirs("data", exist_ok=True)
p = Path("data") / "log.txt"          # 문자열 덧셈 대신 / 연산자
p.write_text("T=25.3\n", encoding="utf-8")
print(p, p.exists(), os.listdir("data"))

### ▶ 예제 5-2 · subprocess — !명령 을 코드 안에서

In [ ]:
r = subprocess.run([sys.executable, "-c", "print(6*7)"], capture_output=True, text=True)
print(r.stdout.strip(), "| returncode:", r.returncode)
# Jetson 에서 시스템 정보를 읽을 때 이 형태를 그대로 쓴다

### ▶ 예제 5-3 · 가짜 시리얼 리플레이
실제 시리얼 루프와 같은 구조 — 데이터 소스만 파일/리스트입니다. 🎬 D3-5 ⑤

In [ ]:
good = bad = skip = 0
for line in STREAM:                     # 실제로는 ser.readline() 자리
    time.sleep(0.05)                    # 샘플링 간격
    if not line.strip():
        skip += 1
        continue                        # timeout 빈 줄 — 오류 아님
    try:
        t, h, d = parse_line(line)
    except (FrameError, ValueError):
        bad += 1
        continue                        # 손상·조각 — 세고 계속
    good += 1
print(f"정상 {good} · 손상 {bad} · 빈 줄 {skip}")   # 3 · 2 · 1

### ▶ 예제 5-4 · 안전 종료 — KeyboardInterrupt

In [ ]:
_k = [0]
def fake_readline():
    _k[0] += 1
    if _k[0] >= 4: raise KeyboardInterrupt   # Ctrl-C 흉내
    return f"$T=2{_k[0]}.0,H=50,D=100*"

processed = 0
while True:
    try:
        line = fake_readline()
        parse_line(line)
        processed += 1
    except KeyboardInterrupt:
        break                            # 루프만 탈출 — 프로그램은 정상 진행
print("지금까지", processed, "줄 처리 후 안전 종료")

#### ✏️ 빈칸 채우기 5-1
`____` 를 채운 뒤 실행하세요. 정답은 아래 **정답 보기** 셀을 실행하면 나타납니다.

In [ ]:
# 리플레이 루프 완성 — 빈 줄은 건너뛰고, 실패는 세고 계속
good = bad = skip = 0
for line in STREAM:
    if not line.strip():
        skip += 1
        ____
    try:
        parse_line(line)
    except (FrameError, ____):
        bad += 1
        continue
    good += 1
print(good, bad, skip)     # 3 2 1

In [ ]:
#@title 🔒 정답 보기 5-1 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA1LTEg4pSA4pSACiAgICAgICAgY29udGludWUKICAgIC4uLgogICAgZXhjZXB0IChGcmFt"
    "ZUVycm9yLCBWYWx1ZUVycm9yKTo="
).decode("utf-8"))

#### ✏️ 괄호 넣기 5-2
1. `ser.readline()` 은 프레임 기호가 아니라 (　　　) 문자에서 자른다 — 그래서 $…* 검증이 필수다.
2. timeout 으로 온 빈 문자열은 오류가 아니므로 `if not line:` 후 (　　　) 한다.
3. 노트북의 `!ls` 를 .py 스크립트에서 대신하는 표준 라이브러리는 (　　　)이다.
4. Ctrl-C 는 (　　　) 예외로 잡아 정상 종료시킬 수 있다.

In [ ]:
#@title 🔒 정답 보기 5-2 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA1LTIg4pSA4pSACjEuIOqwnO2WiShcbikKMi4gY29udGludWUKMy4gc3VicHJvY2Vzcwo0"
    "LiBLZXlib2FyZEludGVycnVwdA=="
).decode("utf-8"))

---
# 6교시 · 종합 설계 — SensorLogger (15:00–15:50)

요구사항: **"측정을 모으고, 파일로 남기고, 다시 불러와 이어서 모을 수 있는 로거"**
메서드 목록: `add / save / load / stats_by_node / report`

🎬 **D3-4 ⑤** to_dict 패턴 · **03** "[Day 2] 함수 호출과 스택" (메서드도 같은 프레임)

### ▶ 예제 6-1 · to_dict / from_dict — 인스턴스 ↔ dict 왕복

In [ ]:
class SensorReading:
    def __init__(self, node_id, t, h=0, d=0):
        self.node_id = node_id; self.t = t; self.h = h; self.d = d
    def __repr__(self):
        return f"Reading({self.node_id}, T={self.t})"
    def is_alert(self):
        return self.t > 35
    def to_dict(self):
        return {"id": self.node_id, "T": self.t, "H": self.h, "D": self.d}
    @classmethod
    def from_dict(cls, d):
        return cls(d["id"], d["T"], d["H"], d["D"])   # cls = 설계도

r  = SensorReading("ESP32-01", 25.3, 60, 120)
r2 = SensorReading.from_dict(r.to_dict())
print(r.to_dict())
print(repr(r) == repr(r2))     # 왕복 성공

### ▶ 예제 6-2 · SensorLogger — add / save / load 라이브 완성

In [ ]:
class SensorLogger:
    def __init__(self, path="readings.json"):
        self.path = path
        self.readings = []

    def add(self, reading):
        self.readings.append(reading)

    def save(self):
        with open(self.path, "w", encoding="utf-8") as f:
            json.dump([r.to_dict() for r in self.readings], f,
                      indent=2, ensure_ascii=False)

    def load(self):
        try:
            with open(self.path, encoding="utf-8") as f:
                self.readings = [SensorReading.from_dict(d) for d in json.load(f)]
        except FileNotFoundError:
            self.readings = []          # 첫 실행 — 빈 채로 시작

lg = SensorLogger("demo.json")
lg.load()                               # 파일이 없어도 죽지 않는다
lg.add(SensorReading("ESP32-01", 25.3, 60, 120))
lg.add(SensorReading("ESP32-02", 36.9, 70, 150))
lg.save()
print(len(lg.readings), open("demo.json", encoding="utf-8").read()[:60], "...")

### ▶ 예제 6-3 · 영속성 확인 — "런타임 재시작" 흉내

In [ ]:
lg2 = SensorLogger("demo.json")         # 완전히 새 로거 (기억이 없다)
lg2.load()                              # 파일에서 복원
lg2.add(SensorReading("ESP32-03", 25.1, 48, 210))
print(len(lg2.readings), lg2.readings)  # 3 — 이어서 쌓인다

#### ✏️ 빈칸 채우기 6-1
`____` 를 채운 뒤 실행하세요. 정답은 아래 **정답 보기** 셀을 실행하면 나타납니다.

In [ ]:
# from_dict 는 왜 classmethod 인가 — 인스턴스가 없는 상태에서 설계도로 만들어야 하므로
class Point:
    def __init__(self, x, y):
        self.x = x; self.y = y
    def to_dict(self):
        return {"x": self.x, "y": self.y}
    @____
    def from_dict(____, d):
        return cls(d["x"], d["y"])

p = Point.from_dict({"x": 1, "y": 2})
print(p.x, p.y)      # 1 2

In [ ]:
#@title 🔒 정답 보기 6-1 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA2LTEg4pSA4pSACkBjbGFzc21ldGhvZApkZWYgZnJvbV9kaWN0KGNscywgZCk6"
).decode("utf-8"))

#### ✏️ 빈칸 채우기 6-2
`____` 를 채운 뒤 실행하세요. 정답은 아래 **정답 보기** 셀을 실행하면 나타납니다.

In [ ]:
# load 의 두 갈래 — 없으면 빈 채로, 있으면 복원
def load_list(path):
    ____:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except ____:
        return []

print(load_list("demo.json") != [], load_list("no_file.json"))   # True []

In [ ]:
#@title 🔒 정답 보기 6-2 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA2LTIg4pSA4pSACnRyeToKICAgIC4uLgpleGNlcHQgRmlsZU5vdEZvdW5kRXJyb3I6"
).decode("utf-8"))

#### ✏️ 괄호 넣기 6-3
1. 인스턴스는 JSON 이 안 되므로 저장 전에 (　　　) 로 dict 로 바꾼다.
2. 복원 시점에는 객체가 없으므로 from_dict 는 (　　　) 로 만들어 (　　　) 로 설계도를 받는다.
3. load 에서 FileNotFoundError 를 잡아 빈 리스트로 시작하는 이유: 첫 실행에 파일이 없는 것은 (　　　)이기 때문.
4. 예외는 오류가 나는 곳이 아니라 (　　　) 곳에서 잡는다 (Day 2 원칙).

In [ ]:
#@title 🔒 정답 보기 6-3 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA2LTMg4pSA4pSACjEuIHRvX2RpY3QoKQoyLiBjbGFzc21ldGhvZCAvIGNscwozLiDsoJXs"
    "g4Eg7Iuc64KY66as7JikCjQuIOuMgOyymO2VoCDsiJgg7J6I64qU"
).decode("utf-8"))

---
# 7교시 · 종합 실습 (16:00–16:50)

실습 워크시트(`Day3_worksheet.ipynb`) 7교시 섹션에서 자동 채점과 함께 진행합니다.
🔒 모범 답안은 8교시 리뷰 때 함께 엽니다.

**과제 A** — 저장되는 센서 로거: 6개 add → save → **새 로거 load** → 2개 추가 → 총 8개, report 에 ⚠
**과제 B** — export_csv + load_robust(.bak 폴백, Counter 집계)

막히면 🎬 **D3-4** (왕복) · **D2-5 트리 뷰어** (저장 구조 확인)

In [ ]:
# 과제 A 뼈대 (실제 작성·채점은 워크시트에서)
# 1) SensorReading: 속성4 + __repr__ + is_alert + to_dict + from_dict
# 2) SensorLogger: add / save / load / stats_by_node / report(⚠ 표시)
# 3) 시나리오: siteA.json — 6개 저장 → 새 로거 복원 → 2개 추가 → 총 8개

In [ ]:
#@title 🔒 정답 보기 7-A — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA3LUEg4pSA4pSACmNsYXNzIFNlbnNvclJlYWRpbmc6CiAgICBkZWYgX19pbml0X18oc2Vs"
    "Ziwgbm9kZV9pZCwgdCwgaD0wLCBkPTApOgogICAgICAgIHNlbGYubm9kZV9pZCA9IG5vZGVfaWQ7IHNlbGYudCA9"
    "IHQ7IHNlbGYuaCA9IGg7IHNlbGYuZCA9IGQKICAgIGRlZiBfX3JlcHJfXyhzZWxmKToKICAgICAgICByZXR1cm4g"
    "ZiJSZWFkaW5nKHtzZWxmLm5vZGVfaWR9LCBUPXtzZWxmLnR9KSIKICAgIGRlZiBpc19hbGVydChzZWxmKToKICAg"
    "ICAgICByZXR1cm4gc2VsZi50ID4gMzUKICAgIGRlZiB0b19kaWN0KHNlbGYpOgogICAgICAgIHJldHVybiB7Imlk"
    "Ijogc2VsZi5ub2RlX2lkLCAiVCI6IHNlbGYudCwgIkgiOiBzZWxmLmgsICJEIjogc2VsZi5kfQogICAgQGNsYXNz"
    "bWV0aG9kCiAgICBkZWYgZnJvbV9kaWN0KGNscywgZCk6CiAgICAgICAgcmV0dXJuIGNscyhkWyJpZCJdLCBkWyJU"
    "Il0sIGRbIkgiXSwgZFsiRCJdKQoKY2xhc3MgU2Vuc29yTG9nZ2VyOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHBh"
    "dGgpOgogICAgICAgIHNlbGYucGF0aCA9IHBhdGg7IHNlbGYucmVhZGluZ3MgPSBbXQogICAgZGVmIGFkZChzZWxm"
    "LCByZWFkaW5nKToKICAgICAgICBzZWxmLnJlYWRpbmdzLmFwcGVuZChyZWFkaW5nKQogICAgZGVmIHNhdmUoc2Vs"
    "Zik6CiAgICAgICAgd2l0aCBvcGVuKHNlbGYucGF0aCwgInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAg"
    "ICAgICAgICBqc29uLmR1bXAoW3IudG9fZGljdCgpIGZvciByIGluIHNlbGYucmVhZGluZ3NdLCBmLCBpbmRlbnQ9"
    "MiwgZW5zdXJlX2FzY2lpPUZhbHNlKQogICAgZGVmIGxvYWQoc2VsZik6CiAgICAgICAgdHJ5OgogICAgICAgICAg"
    "ICB3aXRoIG9wZW4oc2VsZi5wYXRoLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgICAgICAgICAgc2Vs"
    "Zi5yZWFkaW5ncyA9IFtTZW5zb3JSZWFkaW5nLmZyb21fZGljdChkKSBmb3IgZCBpbiBqc29uLmxvYWQoZildCiAg"
    "ICAgICAgZXhjZXB0IEZpbGVOb3RGb3VuZEVycm9yOgogICAgICAgICAgICBzZWxmLnJlYWRpbmdzID0gW10KICAg"
    "IGRlZiBzdGF0c19ieV9ub2RlKHNlbGYpOgogICAgICAgIG91dCA9IHt9CiAgICAgICAgZm9yIG5pZCBpbiBzb3J0"
    "ZWQoe3Iubm9kZV9pZCBmb3IgciBpbiBzZWxmLnJlYWRpbmdzfSk6CiAgICAgICAgICAgIHRzID0gW3IudCBmb3Ig"
    "ciBpbiBzZWxmLnJlYWRpbmdzIGlmIHIubm9kZV9pZCA9PSBuaWRdCiAgICAgICAgICAgIG91dFtuaWRdID0geyJh"
    "dmdfVCI6IHJvdW5kKHN1bSh0cykgLyBsZW4odHMpLCAyKSwKICAgICAgICAgICAgICAgICAgICAgICAgIm1heF9U"
    "IjogbWF4KHRzKSwKICAgICAgICAgICAgICAgICAgICAgICAgImFsZXJ0cyI6IHN1bSgxIGZvciB0IGluIHRzIGlm"
    "IHQgPiAzNSl9CiAgICAgICAgcmV0dXJuIG91dAogICAgZGVmIHJlcG9ydChzZWxmKToKICAgICAgICBwcmludChm"
    "Insnbm9kZSc6MTBzfXsnYXZnX1QnOj43c317J21heF9UJzo+N3N9eydhbGVydHMnOj43c30iKQogICAgICAgIGxp"
    "bmVzID0gW2Yie246MTBzfXtzWydhdmdfVCddOjcuMWZ9e3NbJ21heF9UJ106Ny4xZn17c1snYWxlcnRzJ106N2R9"
    "IgogICAgICAgICAgICAgICAgICsgKCIg4pqgIiBpZiBzWyJhbGVydHMiXSBlbHNlICIiKQogICAgICAgICAgICAg"
    "ICAgIGZvciBuLCBzIGluIHNlbGYuc3RhdHNfYnlfbm9kZSgpLml0ZW1zKCldCiAgICAgICAgZm9yIGwgaW4gbGlu"
    "ZXM6CiAgICAgICAgICAgIHByaW50KGwpCgojIOyLnOuCmOumrOyYpApsb2dnZXJBID0gU2Vuc29yTG9nZ2VyKCJz"
    "aXRlQS5qc29uIikKZm9yIGQgaW4gREFZMl9SRUFESU5HUzoKICAgIGxvZ2dlckEuYWRkKFNlbnNvclJlYWRpbmcu"
    "ZnJvbV9kaWN0KGQpKQpsb2dnZXJBLnNhdmUoKQoKbG9nZ2VyQiA9IFNlbnNvckxvZ2dlcigic2l0ZUEuanNvbiIp"
    "CmxvZ2dlckIubG9hZCgpCmxvZ2dlckIuYWRkKFNlbnNvclJlYWRpbmcoIkVTUDMyLTAxIiwgMzcuMiwgNjYsIDEy"
    "NSkpCmxvZ2dlckIuYWRkKFNlbnNvclJlYWRpbmcoIkVTUDMyLTAzIiwgMjQuMCwgNTAsIDIwNSkpCmxvZ2dlckIu"
    "c2F2ZSgpCmxvZ2dlckIucmVwb3J0KCkKcHJpbnQobGVuKGxvZ2dlckIucmVhZGluZ3MpKSAgICMgOA=="
).decode("utf-8"))

In [ ]:
#@title 🔒 정답 보기 7-B — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA3LUIg4pSA4pSACiMg6rO87KCcIEIg4oCUIGV4cG9ydF9jc3Yg66mU7ISc65OcICsgbG9h"
    "ZF9yb2J1c3QKZGVmIGV4cG9ydF9jc3Yoc2VsZiwgcGF0aCk6CiAgICB3aXRoIG9wZW4ocGF0aCwgInciLCBuZXds"
    "aW5lPSIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIHcgPSBjc3Yud3JpdGVyKGYpCiAgICAgICAg"
    "dy53cml0ZXJvdyhbIm5vZGUiLCAiVCIsICJIIiwgIkQiLCAiYWxlcnQiXSkKICAgICAgICBmb3IgciBpbiBzZWxm"
    "LnJlYWRpbmdzOgogICAgICAgICAgICB3LndyaXRlcm93KFtyLm5vZGVfaWQsIHIudCwgci5oLCByLmQsIHIuaXNf"
    "YWxlcnQoKV0pClNlbnNvckxvZ2dlci5leHBvcnRfY3N2ID0gZXhwb3J0X2NzdiAgICMgKO2BtOuemOyKpCDslYjs"
    "l5Ag7KeB7KCRIOygleydmO2VtOuPhCDrj5nsnbwpCgplcnJfY291bnRzID0gQ291bnRlcigpCmRlZiBsb2FkX3Jv"
    "YnVzdChwYXRoKToKICAgIGZvciBwIGluIChwYXRoLCBwYXRoICsgIi5iYWsiKToKICAgICAgICB0cnk6CiAgICAg"
    "ICAgICAgIHdpdGggb3BlbihwLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgICAgICAgICAgcmV0dXJu"
    "IGpzb24ubG9hZChmKQogICAgICAgIGV4Y2VwdCAoanNvbi5KU09ORGVjb2RlRXJyb3IsIEZpbGVOb3RGb3VuZEVy"
    "cm9yKSBhcyBlOgogICAgICAgICAgICBlcnJfY291bnRzW3R5cGUoZSkuX19uYW1lX19dICs9IDEKICAgIHJldHVy"
    "biBbXQoKbG9nZ2VyQi5leHBvcnRfY3N2KCJzaXRlQS5jc3YiKQpkYXRhMSA9IGxvYWRfcm9idXN0KCJicm9rZW4u"
    "anNvbiIpICAgICAjIC5iYWsg7JeQ7IScIOuzteq1rApkYXRhMiA9IGxvYWRfcm9idXN0KCJob3BlbGVzcy5qc29u"
    "IikgICAjIOuRmCDri6Qg7Iuk7YyoIOKGkiBbXQpwcmludChsZW4oZGF0YTEpLCBkYXRhMiwgZGljdChlcnJfY291"
    "bnRzKSk="
).decode("utf-8"))

---
# 8교시 · 정리 (17:00–17:50)

## 오늘의 여섯 문장
1. 클래스는 설계도, 인스턴스는 데이터 — self 는 "지금 이 인스턴스" (D3-1)
2. 조회는 인스턴스 → 클래스, 자식 __init__ 첫 줄은 super() (D3-1·D3-2)
3. "w" 는 지우고 "a" 는 잇는다 — with 가 close 를 보장 (D3-3)
4. dump/load 는 파일, dumps/loads 는 문자열 — 복원하면 타입이 돌아온다 (D3-4)
5. 인스턴스는 to_dict 로 저장, from_dict(classmethod) 로 복원 (D3-4·6교시)
6. readline 은 개행에서 자를 뿐 — 검증·건너뛰기·안전 종료가 견고한 수집기 (D3-5)

- Day 3 퀴즈: Google Forms 링크
- 제출: `Day3_이름.ipynb` (제출 요약 셀 실행) + `siteA.json` + 개념 워크시트 캡처
- **Day 4 예고**: 마무리 프로젝트 — 오늘의 로거에 실시간 스트림 리플레이와 통계 리포트/시각화를 붙여 완성하고 발표합니다. Jetson 주간으로 가는 다리입니다.